# Seq2Seq Machine Translation: Encoder Architecture Comparison

French → English neural machine translation with five encoder designs, implemented in PyTorch and
evaluated with ROUGE-1 and ROUGE-2 F1 on a held-out test set.

*Coursework for MH6812, Nanyang Technological University. The data pipeline and the baseline GRU
encoder–decoder are adapted from the PyTorch tutorial
["NLP From Scratch: Translation with a Sequence to Sequence Network and Attention"](https://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html).*

| Exp | Encoder | Decoder |
|---|---|---|
| 1 | GRU (baseline) | GRU |
| 2 | LSTM | LSTM |
| 3 | Bi-LSTM (final states summed) | LSTM |
| 4 | GRU | GRU with additive attention |
| 5 | Transformer encoder (2 layers, mean-pooled) | GRU |

The results reported in the README come from the original run; its full output is in
`results/original_run_log.txt`. This notebook keeps the same models and training procedure but
corrects three evaluation issues (listed in the README), so re-run ROUGE scores will be somewhat
higher than the reported ones. Running all five experiments takes about 90 minutes on a
Colab T4 GPU. Set `SMOKE_TEST = True` to check that everything runs in a few minutes on a CPU.

## 1. Setup

In [ ]:
import math
import os
import random
import re
import time
import unicodedata
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from sklearn.model_selection import train_test_split
from torchmetrics.text.rouge import ROUGEScore
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}")

In [ ]:
# Shared settings (identical for all five experiments)
SOS_token = 0
EOS_token = 1
MAX_LENGTH = 15                # sentences must have fewer than 15 words
HIDDEN_SIZE = 256
EPOCHS = 3
LEARNING_RATE = 0.01           # plain SGD
TEACHER_FORCING_RATIO = 0.5    # probability of feeding ground-truth tokens for a whole sentence
PRINT_EVERY = 2000             # report average loss every 2,000 training pairs

SMOKE_TEST = False             # True: tiny subset and 1 epoch, to check the code runs

## 2. Data

Tatoeba English–French sentence pairs from [manythings.org](http://www.manythings.org/anki/).
Pairs are normalised to lower-case ASCII, kept only if both sides have fewer than 15 words and the
English side starts with a simple subject–verb prefix ("i am", "he is", "you re", …), and reversed so
that **French is the source and English the target**.

In [ ]:
DATA_PATH = "data/eng-fra.txt"

if not os.path.exists(DATA_PATH):
    os.makedirs("data", exist_ok=True)
    request = urllib.request.Request("http://www.manythings.org/anki/fra-eng.zip",
                                     headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request) as response, open("fra-eng.zip", "wb") as f:
        f.write(response.read())
    with zipfile.ZipFile("fra-eng.zip") as z:
        z.extract("fra.txt", "data")
    os.replace("data/fra.txt", DATA_PATH)
print("Data file:", DATA_PATH)

In [ ]:
class Lang:
    """Word-level vocabulary for one language."""
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2

    def addSentence(self, sentence):
        for word in sentence.split(" "):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1


def unicodeToAscii(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")


def normalizeString(s):
    """Lower-case, strip accents, separate . ! ? and drop all other non-letters."""
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s


def readLangs(lang1, lang2, reverse=False):
    lines = open(f"data/{lang1}-{lang2}.txt", encoding="utf-8").read().strip().split("\n")
    # Each line is: English <tab> French <tab> attribution; keep the first two fields
    pairs = [[normalizeString(s) for s in line.split("\t")[:2]] for line in lines]
    if reverse:
        pairs = [[p[1], p[0]] for p in pairs]
        input_lang, output_lang = Lang(lang2), Lang(lang1)
    else:
        input_lang, output_lang = Lang(lang1), Lang(lang2)
    return input_lang, output_lang, pairs


ENG_PREFIXES = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re ",
)


def filterPair(p):
    # p = [French, English] after reversal; the prefix filter applies to the English side
    return (len(p[0].split(" ")) < MAX_LENGTH and len(p[1].split(" ")) < MAX_LENGTH
            and p[1].startswith(ENG_PREFIXES))


def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print(f"Read {len(pairs)} sentence pairs")
    pairs = [p for p in pairs if filterPair(p)]
    print(f"Trimmed to {len(pairs)} sentence pairs")
    # Note: vocabularies are built from all filtered pairs (train and test), as in the original
    # run, so the test set contains no out-of-vocabulary words.
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print(f"Vocabulary: {input_lang.name} {input_lang.n_words}, {output_lang.name} {output_lang.n_words}")
    return input_lang, output_lang, pairs


# Source = French, target = English
input_lang, output_lang, pairs = prepareData("eng", "fra", reverse=True)
print("Example:", random.choice(pairs))

In [ ]:
X = [p[0] for p in pairs]
y = [p[1] for p in pairs]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
train_pairs = list(zip(X_train, y_train))
test_pairs = list(zip(X_test, y_test))

if SMOKE_TEST:
    train_pairs, test_pairs, EPOCHS, PRINT_EVERY = train_pairs[:300], test_pairs[:50], 1, 100

print(f"Train pairs: {len(train_pairs)}, Test pairs: {len(test_pairs)}")


def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(" ")]


def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence) + [EOS_token]
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)


def tensorsFromPair(pair):
    return tensorFromSentence(input_lang, pair[0]), tensorFromSentence(output_lang, pair[1])

## 3. Models

The recurrent encoders read the source sentence one token at a time (batch size 1). Each encoder
exposes `output_size`, the width of the per-token outputs stored for attention.

In [ ]:
class EncoderRNN(nn.Module):
    """Single-layer GRU encoder (baseline)."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.output_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        return self.gru(embedded, hidden)

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)


class EncoderLSTM(nn.Module):
    """Single-layer LSTM encoder."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.output_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        return self.lstm(embedded, hidden)

    def initHidden(self):
        return (torch.zeros(1, 1, self.hidden_size, device=device),
                torch.zeros(1, 1, self.hidden_size, device=device))


class EncoderBiLSTM(nn.Module):
    """Bidirectional LSTM encoder, called one token at a time.

    Limitation (kept to match the reported run): because each call sees a length-1 sequence and
    both directions carry their state forward through the sentence, the "backward" direction never
    sees later tokens. This encoder therefore behaves like two forward LSTMs rather than a true
    Bi-LSTM. Passing the whole sentence in one call would fix this.
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.output_size = 2 * hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, bidirectional=True)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        return self.lstm(embedded, hidden)

    def initHidden(self):
        return (torch.zeros(2, 1, self.hidden_size, device=device),
                torch.zeros(2, 1, self.hidden_size, device=device))


class TransformerEncoder(nn.Module):
    """Two-layer Transformer encoder with learnable positional embeddings.

    Reads the whole sentence at once and returns per-token outputs plus their mean, which
    initialises the GRU decoder's hidden state.
    """
    def __init__(self, input_size, hidden_size, num_layers=2, nhead=8, max_len=100):
        super().__init__()
        self.hidden_size = hidden_size
        self.output_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.pos_embedding = nn.Embedding(max_len, hidden_size)
        layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=nhead,
                                           dim_feedforward=4 * hidden_size, dropout=0.1)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers,
                                                 enable_nested_tensor=False)

    def forward(self, input_tensor):
        seq_len = input_tensor.size(0)                                  # input: (seq_len, 1)
        positions = torch.arange(seq_len, device=input_tensor.device).unsqueeze(1)
        x = self.embedding(input_tensor) + self.pos_embedding(positions)  # (seq_len, 1, hidden)
        enc_out = self.transformer(x)                                   # (seq_len, 1, hidden)
        context = enc_out.mean(dim=0, keepdim=True)                     # (1, 1, hidden)
        return enc_out, context

In [ ]:
class Decoder(nn.Module):
    """GRU decoder (used with the GRU and Transformer encoders)."""
    def __init__(self, hidden_size, output_size):
        super().__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, input, hidden):
        output = F.relu(self.embedding(input).view(1, 1, -1))
        output, hidden = self.gru(output, hidden)
        return F.log_softmax(self.out(output[0]), dim=1), hidden


class DecoderLSTM(nn.Module):
    """LSTM decoder (used with the LSTM and Bi-LSTM encoders)."""
    def __init__(self, hidden_size, output_size):
        super().__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, input, hidden):
        output = F.relu(self.embedding(input).view(1, 1, -1))
        output, hidden = self.lstm(output, hidden)
        return F.log_softmax(self.out(output[0]), dim=1), hidden


class AdditiveAttention(nn.Module):
    """Additive (Bahdanau-style, concat) attention: score_i = v^T tanh(W [s; h_i]).

    Scores for all source positions are computed in one vectorised step.
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: (1, 1, hidden); encoder_outputs: (MAX_LENGTH, hidden)
        n = encoder_outputs.size(0)
        s = decoder_hidden.squeeze(0).repeat(n, 1)                          # (n, hidden)
        energy = torch.tanh(self.attn(torch.cat([s, encoder_outputs], dim=1)))
        scores = self.v(energy).squeeze(1)                                  # (n,)
        weights = F.softmax(scores, dim=0).view(1, 1, -1)                   # (1, 1, n)
        context = torch.bmm(weights, encoder_outputs.unsqueeze(0))          # (1, 1, hidden)
        return weights, context


class DecoderWithAttention(nn.Module):
    """GRU decoder with additive attention.

    At each step the GRU is updated first; its new hidden state queries the encoder outputs
    (Luong-style placement), and the output word is predicted from the attention context vector.
    The encoder-output buffer is zero-padded to MAX_LENGTH and the padded slots are not masked.
    """
    def __init__(self, hidden_size, output_size):
        super().__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.attention = AdditiveAttention(hidden_size)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, input, hidden, encoder_outputs):
        output = F.relu(self.embedding(input).view(1, 1, -1))
        _, hidden = self.gru(output, hidden)
        _, context = self.attention(hidden, encoder_outputs)
        return F.log_softmax(self.out(context[0]), dim=1), hidden

## 4. Training and inference

In [ ]:
def encode(encoder, input_tensor, encoder_type):
    """Run the encoder; return (encoder_outputs, initial decoder hidden state)."""
    if encoder_type == "transformer":
        return encoder(input_tensor)

    encoder_hidden = encoder.initHidden()
    encoder_outputs = torch.zeros(MAX_LENGTH, encoder.output_size, device=device)
    for ei in range(input_tensor.size(0)):
        encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] = encoder_output[0, 0]

    # Bi-LSTM: merge the two directions' final states by summation for the 1-layer decoder
    if isinstance(encoder_hidden, tuple) and encoder_hidden[0].size(0) == 2:
        h, c = encoder_hidden
        encoder_hidden = (h.sum(dim=0, keepdim=True), c.sum(dim=0, keepdim=True))
    return encoder_outputs, encoder_hidden


def decode_step(decoder, decoder_input, decoder_hidden, encoder_outputs, use_attention):
    if use_attention:
        return decoder(decoder_input, decoder_hidden, encoder_outputs)
    return decoder(decoder_input, decoder_hidden)


def train_step(input_tensor, target_tensor, encoder, decoder, encoder_optimizer,
               decoder_optimizer, criterion, encoder_type, use_attention):
    """One SGD update on a single sentence pair; returns the loss per target token."""
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    encoder_outputs, decoder_hidden = encode(encoder, input_tensor, encoder_type)
    decoder_input = torch.tensor([[SOS_token]], device=device)
    target_length = target_tensor.size(0)
    use_teacher_forcing = random.random() < TEACHER_FORCING_RATIO

    loss = 0
    for di in range(target_length):
        decoder_output, decoder_hidden = decode_step(
            decoder, decoder_input, decoder_hidden, encoder_outputs, use_attention)
        loss += criterion(decoder_output, target_tensor[di])
        if use_teacher_forcing:
            decoder_input = target_tensor[di].view(1, 1)
        else:
            decoder_input = decoder_output.topk(1)[1].detach().view(1, 1)
            if decoder_input.item() == EOS_token:
                break

    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
    return loss.item() / target_length


def train_model(encoder, decoder, encoder_type, use_attention, epochs=EPOCHS):
    """Train with SGD for a number of epochs; return the loss history.

    The history holds the average loss over each window of PRINT_EVERY training pairs.
    """
    encoder_optimizer = optim.SGD(encoder.parameters(), lr=LEARNING_RATE)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=LEARNING_RATE)
    criterion = nn.NLLLoss()

    history, window_loss, it = [], 0.0, 0
    n_iters = len(train_pairs) * epochs
    start = time.time()
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        for pair in train_pairs:
            input_tensor, target_tensor = tensorsFromPair(pair)
            window_loss += train_step(input_tensor, target_tensor, encoder, decoder,
                                      encoder_optimizer, decoder_optimizer, criterion,
                                      encoder_type, use_attention)
            it += 1
            if it % PRINT_EVERY == 0:
                history.append((it, window_loss / PRINT_EVERY))
                window_loss = 0.0
                elapsed = time.time() - start
                print(f"  {it}/{n_iters}  loss={history[-1][1]:.4f}  "
                      f"elapsed {elapsed / 60:.1f} min")
    return history


def translate(encoder, decoder, sentence, encoder_type, use_attention):
    """Greedy decoding of one source sentence; returns the list of output words."""
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        encoder_outputs, decoder_hidden = encode(encoder, input_tensor, encoder_type)
        decoder_input = torch.tensor([[SOS_token]], device=device)
        words = []
        for _ in range(MAX_LENGTH):
            decoder_output, decoder_hidden = decode_step(
                decoder, decoder_input, decoder_hidden, encoder_outputs, use_attention)
            topi = decoder_output.topk(1)[1]
            if topi.item() == EOS_token:
                words.append("<EOS>")
                break
            words.append(output_lang.index2word[topi.item()])
            decoder_input = topi.detach().view(1, 1)
        return words


def inference(encoder, decoder, testing_pairs, encoder_type, use_attention):
    encoder.eval(); decoder.eval()
    references, predictions = [], []
    for src, ref in tqdm(testing_pairs, leave=False):
        words = translate(encoder, decoder, src, encoder_type, use_attention)
        # Drop the end-of-sentence marker: ROUGE would otherwise count it as an extra word
        predictions.append(" ".join(w for w in words if w != "<EOS>"))
        references.append(ref)
    encoder.train(); decoder.train()
    return references, predictions


def eval_rouge(references, predictions):
    """ROUGE-1/2 scores. torchmetrics expects (predictions, references) in that order.

    Only ROUGE-1 and ROUGE-2 are computed; this avoids ROUGE-Lsum, which needs NLTK's
    sentence tokenizer, and gives the same ROUGE-1/2 values as the default settings.
    """
    scores = ROUGEScore(rouge_keys=("rouge1", "rouge2"))(predictions, references)
    for k in ("rouge1", "rouge2"):
        print(f"  {k.upper()}  F1 {scores[k + '_fmeasure']:.4f} | "
              f"precision {scores[k + '_precision']:.4f} | recall {scores[k + '_recall']:.4f}")
    return {k: float(v) for k, v in scores.items()}

## 5. Experiments

All models use hidden size 256, SGD (learning rate 0.01), teacher forcing ratio 0.5 and 3 epochs.
Training pairs are visited in the same fixed order each epoch.

In [ ]:
EXPERIMENTS = [
    # name,               encoder class,      decoder class,        encoder type,  attention
    ("GRU (baseline)",      EncoderRNN,         Decoder,              "rnn",         False),
    ("LSTM",                EncoderLSTM,        DecoderLSTM,          "rnn",         False),
    ("Bi-LSTM",             EncoderBiLSTM,      DecoderLSTM,          "rnn",         False),
    ("GRU + attention",     EncoderRNN,         DecoderWithAttention, "rnn",         True),
    ("Transformer encoder", TransformerEncoder, Decoder,              "transformer", False),
]

results, histories, trained = [], {}, {}
for name, enc_cls, dec_cls, enc_type, use_attn in EXPERIMENTS:
    print("=" * 70 + f"\n{name}\n" + "=" * 70)
    encoder = enc_cls(input_lang.n_words, HIDDEN_SIZE).to(device)
    decoder = dec_cls(HIDDEN_SIZE, output_lang.n_words).to(device)
    t0 = time.time()
    histories[name] = train_model(encoder, decoder, enc_type, use_attn)
    train_minutes = (time.time() - t0) / 60
    refs, preds = inference(encoder, decoder, test_pairs, enc_type, use_attn)
    scores = eval_rouge(refs, preds)
    trained[name] = (encoder, decoder, enc_type, use_attn)
    results.append({
        "Model": name,
        "ROUGE-1 F1": scores["rouge1_fmeasure"],
        "ROUGE-2 F1": scores["rouge2_fmeasure"],
        "Final loss": histories[name][-1][1] if histories[name] else float("nan"),
        "Train time (min)": train_minutes,
    })

## 6. Results

In [ ]:
df = pd.DataFrame(results)
base = df.loc[0]
df["Δ ROUGE-1 vs baseline"] = (df["ROUGE-1 F1"] / base["ROUGE-1 F1"] - 1).map("{:+.1%}".format)
df["Δ ROUGE-2 vs baseline"] = (df["ROUGE-2 F1"] / base["ROUGE-2 F1"] - 1).map("{:+.1%}".format)
print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

os.makedirs("results", exist_ok=True)
df.to_csv("results/results.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.6))
for (name, hist), marker in zip(histories.items(), ["o", "^", "v", "s", "D"]):
    if hist:
        ax.plot([h[0] for h in hist], [h[1] for h in hist], marker=marker, markersize=4,
                markevery=3, linewidth=2, label=f"{name} ({hist[-1][1]:.2f})")
for e in range(1, EPOCHS):
    ax.axvline(e * len(train_pairs), color="grey", linestyle=":", linewidth=1)
ax.set_xlabel("Training iterations (one sentence pair each)")
ax.set_ylabel(f"Average NLL loss per token ({PRINT_EVERY:,}-iteration window)")
ax.set_title("Training loss by encoder architecture (French → English)", loc="left")
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("results/loss_comparison_rerun.png", dpi=160)
plt.show()

In [ ]:
# Translation examples: baseline vs attention on random test sentences
base_model, attn_model = trained["GRU (baseline)"], trained["GRU + attention"]
for i, (src, ref) in enumerate(random.sample(test_pairs, 5), 1):
    print(f"--- Example {i} ---")
    print(f"Source (French):   {src}")
    print(f"Reference:         {ref}")
    print(f"GRU baseline:      {' '.join(translate(base_model[0], base_model[1], src, 'rnn', False))}")
    print(f"GRU + attention:   {' '.join(translate(attn_model[0], attn_model[1], src, 'rnn', True))}")